In [1]:
import pandas as pd
import glob

# 1. Cargar CSV
archivos = glob.glob("../data/*2025*.csv") + glob.glob("data/*2025*.csv")
ruta_csv = archivos[0]
print(f"Cargando: {ruta_csv}")

df = pd.read_csv(ruta_csv, sep=",", encoding="utf-8", low_memory=False)
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

print(f"Registros originales: {df.shape[0]:,}")

Cargando: ../data/sistema-unico-de-atencion-ciudadana-2025.csv
Registros originales: 1,458,570


In [2]:
import numpy as np

# 1. Target binario (1 = Cerrado, 0 = Abierto/Otro)
df['target'] = (df['estado_general'].astype(str).str.lower().str.strip() == 'cerrado').astype(int)

# 2. Variables de fecha y hora
df['fecha_ingreso'] = pd.to_datetime(df['fecha_ingreso'], errors='coerce')
df['mes'] = df['fecha_ingreso'].dt.month
df['dia_semana'] = df['fecha_ingreso'].dt.day_name()
df['es_fin_de_semana'] = df['fecha_ingreso'].dt.dayofweek.isin([5, 6]).astype(int)

df['hora_dt'] = pd.to_datetime(df['hora_ingreso'].str.strip(), format='%I:%M:%S %p', errors='coerce')
df['hora_dt'] = df['hora_dt'].fillna(pd.to_datetime(df['hora_ingreso'].str.strip(), format='%H:%M:%S', errors='coerce'))
df['hora'] = df['hora_dt'].dt.hour

# 3. Imputación y normalización
df['comuna'] = df['comuna'].fillna(-1).astype(int).astype(str)
df['barrio'] = df['barrio'].fillna('DESCONOCIDO').astype(str).str.upper().str.strip()
df['canal'] = df['canal'].fillna('DESCONOCIDO').astype(str).str.upper().str.strip()
df['categoria'] = df['categoria'].fillna('OTRAS').astype(str).str.upper().str.strip()
df['tipo'] = df['tipo'].fillna('DESCONOCIDO').astype(str).str.upper().str.strip()
df['prestacion'] = df['prestacion'].fillna('DESCONOCIDO').astype(str).str.upper().str.strip()

# 4. Filtrar columnas finales
columnas_modelo = [
    'categoria', 'prestacion', 'tipo', 'canal', 
    'comuna', 'barrio', 'mes', 'dia_semana', 
    'es_fin_de_semana', 'hora', 'target'
]

df_modelo = df[columnas_modelo].dropna(subset=['target', 'mes', 'hora']).copy()
print(f"Registros depurados: {len(df_modelo):,}")

Registros depurados: 1,458,570


In [3]:
# Guardar dataset procesado en CSV comprimido
ruta_salida = "../data/suaci_2025_preparado.csv.gz"
df_modelo.to_csv(ruta_salida, index=False, compression="gzip")
print(f"✅ Dataset guardado exitosamente en: {ruta_salida}")

✅ Dataset guardado exitosamente en: ../data/suaci_2025_preparado.csv.gz
